# Probabilistic forecast evaluation

This notebook demonstrates evaluation of **quantile (probabilistic) forecasts**.
We work with a 9-quantile forecast and show:
* CRPS and fairCRPS
* Quantile (pinball) loss
* Brier score
* Probability Integral Transform (QQ plot)
* Resolution / sharpness

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from performance import ForecastPerformance
from performance.metrics.probabilistic import quantile_loss

## 1  Synthetic data

In [ ]:
rng = np.random.default_rng(1)

dates = pd.date_range("2018-01-01", periods=365 * 3, freq="D")
t = np.arange(len(dates))

# Seasonal reference
reference = pd.Series(
    10 + 8 * np.sin(2 * np.pi * t / 365) + rng.normal(0, 1, len(t)),
    index=dates,
    name="Reference",
)

## 2  Build a 9-quantile forecast

Each quantile is the reference perturbed by a Normal draw whose spread
increases with the distance from the median.

In [ ]:
QUANTILE_LEVELS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

# Generate sorted quantiles: use the quantiles of a Normal(ref, sigma)
from scipy.stats import norm

sigma = 1.5  # forecast spread
quantile_array = np.stack(
    [reference.values + norm.ppf(q) * sigma for q in QUANTILE_LEVELS], axis=1
)

prob_df = pd.DataFrame(
    quantile_array,
    index=dates,
    columns=pd.MultiIndex.from_product(
        [[pd.Timedelta("0D")], QUANTILE_LEVELS],
        names=["Leadtime", "Probability"],
    ),
)

fp = ForecastPerformance(reference)
fp.add_by_production_date(prob_df, name="prob_fcst")
print("Simulation type:", fp.simulations["prob_fcst"]["simulationType"])

## 3  CRPS

In [ ]:
lt = pd.Timedelta("0D")

crps_val = fp.CRPS("prob_fcst", leadtime=lt)
print(f"Mean CRPS: {crps_val:.4f}")

## 4  Fair CRPS

`fairCRPS` uses the same formula as CRPS but returns a skill score
relative to a reference simulation (here the climatology would be added;
we skip it and compare the forecast with itself to show a skill score of 1).

In [ ]:
fair_crps_val = fp.fairCRPS("prob_fcst", reference="prob_fcst", leadtime=lt)
print(f"Fair CRPSS (self-reference): {fair_crps_val:.4f}")

## 5  Quantile (pinball) loss

In [ ]:
sim_data = fp.simulations["prob_fcst"]["data"]
# Select the leadtime slice and align with the reference
tmp = pd.concat((sim_data.xs(lt, level="Leadtime", axis=1), reference), axis=1).dropna()
sim_vals = tmp.drop("Reference", axis=1).values
target_vals = tmp["Reference"].values

ql = quantile_loss(sim_vals, np.array(QUANTILE_LEVELS), target_vals)
print(f"Mean quantile loss: {ql:.4f}")

## 6  Brier score

In [ ]:
# Brier score for P(X > median)
threshold = float(reference.median())
bs = fp.BrierS("prob_fcst", threshold=threshold, leadtime=lt)
print(f"Brier score (threshold={threshold:.1f}): {bs:.4f}")

## 7  Probability Integral Transform (QQ plot)

A well-calibrated probabilistic forecast has a uniform PIT distribution,
which appears as a straight diagonal line on the QQ plot.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
fp.QQPlot("prob_fcst", leadtimes=[lt], ax=ax)
ax.set_title("QQ plot — prob_fcst")
plt.tight_layout();

## 8  Reliability and resolution

In [ ]:
rel = fp.reliability("prob_fcst", [lt])
res = fp.resolution("prob_fcst", [lt])
print(f"Reliability (closer to 1 is better): {rel:.4f}")
print(f"Resolution / sharpness (higher is better): {res:.4f}")